# Entraînement du DÉTECTEUR de lésions — YOLOv8-seg (Colab GPU)

Pipeline : dataset CBIS → dataset YOLO-seg (masques ROI → polygones, split officiel par patient)
→ **baseline du modèle déployé** → entraînement (1280px / `yolov8s-seg`) → évaluation par classe
→ comparaison → publication.

**Contexte** : audit `ifar/docs/AUDIT_MODELES_IA.md` — détecteur déployé à **mAP50 ≈ 0,10**, rate
la plupart des lésions (surtout les microcalcifications, effacées à 640px). Objectif : recall élevé.
Runtime **GPU** requis.

## 1. Paramètres

In [ ]:
# ⚠️ Branche portant le builder YOLO-seg, l'éval détecteur et la porte SaMD.
#    Mettre 'main' UNIQUEMENT une fois cette branche fusionnée.
MLOPS_BRANCH = 'claude/ifar-classifier-training-i6pohk'

HF_DATASET_REPO = 'Mailcoding/ifar-mammo-rois'   # dataset HF privé (JPEG CBIS + CSV). Ou utiliser Drive (cellule 4b).
DEPLOYED_SPACE  = 'Mailcoding/ifar_ml'           # Space ml-service → baseline yolov8_seg.pt déployé
HF_MODEL_REPO   = 'Mailcoding/ifar-mammo-detector'
VERSION = 'v0.1.0'
TRAIN_CSVS = [('mass_case_description_train_set.csv','mass'), ('calc_case_description_train_set.csv','calc')]
VAL_CSVS   = [('mass_case_description_test_set.csv','mass'),  ('calc_case_description_test_set.csv','calc')]
YOLO_DIR = '/content/yolo_seg'   # dataset YOLO-seg généré
# (Optionnel) Généralisation VinDr-Mammo — laisser None pour CBIS seul.
VINDR_ANNOTATIONS = None   # ex. '/content/vindr/finding_annotations.csv'
VINDR_IMAGES = None        # ex. '/content/vindr/images' (png nommés image_id.png)

## 2. Installer `ifar-mlops` (clone auto)

In [ ]:
import os
MLOPS_SRC = '/content/ifar-mlops'

if os.path.exists(f'{MLOPS_SRC}/.git'):
    !cd "{MLOPS_SRC}" && git fetch -q origin "{MLOPS_BRANCH}" && git checkout -q "{MLOPS_BRANCH}" && git reset -q --hard "origin/{MLOPS_BRANCH}"
else:
    # Repo PUBLIC : direct. Repo PRIVÉ : insérer un PAT → https://<PAT>@github.com/...
    !git clone -q --branch "{MLOPS_BRANCH}" https://github.com/mailcoding/ifar-mlops.git "{MLOPS_SRC}"

assert os.path.exists(f'{MLOPS_SRC}/pyproject.toml'), (
    f'{MLOPS_SRC}/pyproject.toml introuvable. Branche inexistante, ou repo privé → PAT requis :\n'
    f'  !git clone --branch {MLOPS_BRANCH} https://<PAT>@github.com/mailcoding/ifar-mlops.git {MLOPS_SRC}')

!pip install -q "{MLOPS_SRC}[train]"
%cd "{MLOPS_SRC}"
!git log --oneline -1

# Vérifie que le code installé est à jour (échec explicite plutôt qu'ImportError plus loin).
import torch
from mlops.datasets.build_yolo_seg_dataset import build          # noqa: F401
from mlops.eval.detector_metrics import evaluate_detector        # noqa: F401
import mlops.validation.gate                                     # noqa: F401
print('mlops OK | builder YOLO-seg ✔ | éval détecteur ✔ | porte SaMD ✔ | CUDA', torch.cuda.is_available())
if not torch.cuda.is_available():
    print("⚠️ Pas de GPU : Exécution → Modifier le type d'exécution → GPU (T4).")

## 3. Authentification Hugging Face

In [ ]:
import os
from huggingface_hub import login
os.environ['HF_TOKEN'] = 'hf_xxx'   # token HF (read dataset+space / write publication)
login(os.environ['HF_TOKEN'])

## 4a. Dataset depuis Hugging Face (snapshot du repo privé)

In [ ]:
from huggingface_hub import snapshot_download
DATA_DIR = snapshot_download(repo_id=HF_DATASET_REPO, repo_type='dataset', local_dir='/content/data')
IMAGES_ROOT = DATA_DIR   # racine des JPEG (arbre CBIS)
CSV_DIR = DATA_DIR       # dossier des CSV de cas
print('Dataset :', DATA_DIR)

## 4b. (Alternative) Dataset depuis Google Drive
Décommenter si les données sont dans Drive (ex. dossier `cbis_ddsm/`) plutôt que sur HF.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# BASE = '/content/drive/MyDrive/cbis_ddsm/cbis_ddsm'
# IMAGES_ROOT = f'{BASE}/jpeg'
# CSV_DIR = f'{BASE}/csv'

## 5. Construire le dataset YOLO-seg (masques ROI → polygones)

In [ ]:
import os
from mlops.datasets.build_yolo_seg_dataset import build

train_csvs = [(os.path.join(CSV_DIR, c), t) for c, t in TRAIN_CSVS]
val_csvs   = [(os.path.join(CSV_DIR, c), t) for c, t in VAL_CSVS]
report = build(train_csvs, val_csvs, IMAGES_ROOT, YOLO_DIR, link=True, materialize=True)
assert not report['patient_leak'], f"FUITE PATIENT : {report['patient_leak'][:5]}"
tr, va = report['train'], report['val']
print(f"JPEG indexés : {report['n_jpeg_indexed']}")
print(f"train : {tr['materialized']} | lésions {tr['n_lesions']} | {len(tr['patients'])} patients")
print(f"val   : {va['materialized']} | lésions {va['n_lesions']} | {len(va['patients'])} patients")
assert tr['materialized']['written'] and va['materialized']['written'], 'split vide — vérifier IMAGES_ROOT/CSV_DIR'
DATA_YAML = report['data_yaml']; print('data.yaml :', DATA_YAML)

## 5b. (Optionnel) Fusionner VinDr-Mammo — généralisation
FFDM moderne + gros volume de findings. Se fond dans le **même** dataset YOLO-seg (préfixe `vindr_`).
VinDr sert le **détecteur** (localisation), pas le classifieur (pas de vérité histologique).

In [ ]:
if VINDR_ANNOTATIONS:
    from mlops.datasets.build_vindr_yolo_dataset import build as build_vindr
    vrep = build_vindr(VINDR_ANNOTATIONS, VINDR_IMAGES, YOLO_DIR, mode='seg', materialize=True)
    assert not vrep['study_leak'], f"FUITE ÉTUDE VinDr : {vrep['study_leak'][:5]}"
    print('VinDr fusionné :', vrep['materialized'])
    print('  findings train', vrep['train']['n_findings'], '| val', vrep['val']['n_findings'])
else:
    print('VinDr non configuré (VINDR_ANNOTATIONS=None) — dataset CBIS seul.')

## 6. Baseline — évaluer le modèle DÉPLOYÉ (`yolov8_seg.pt`)
Chiffre l'existant sur CE jeu de validation, pour comparer honnêtement au nouveau modèle.

In [ ]:
from huggingface_hub import hf_hub_download
from mlops.eval.detector_metrics import evaluate_detector, summarize

deployed = hf_hub_download(repo_id=DEPLOYED_SPACE, repo_type='space', filename='models/yolov8_seg.pt')
baseline = evaluate_detector(deployed, DATA_YAML, imgsz=1280, target_recall=0.90)
print('=== BASELINE (modèle déployé) ==='); print(summarize(baseline))

## 7. Entraîner (1280px / `yolov8s-seg`, via la config renforcée)

In [ ]:
import yaml, copy, subprocess
base = yaml.safe_load(open('configs/mammo_detector.yaml'))
cfg = copy.deepcopy(base)
cfg['data']['yaml'] = DATA_YAML
cfg['export']['out_dir'] = '/content/artifacts/mammo-det'
cfg['export']['version'] = VERSION
p = '/content/config_det.yaml'; yaml.safe_dump(cfg, open(p, 'w'))
print('imgsz', cfg['train']['imgsz'], '| base', cfg['model']['base'])
subprocess.run(['python', '-m', 'mlops.train.train_mammo_detector', '--config', p], check=True)

## 8. Évaluer le nouveau modèle

In [ ]:
NEW_WEIGHTS = f"/content/artifacts/mammo-det/yolov8_seg.pt"
new = evaluate_detector(NEW_WEIGHTS, DATA_YAML, imgsz=1280, target_recall=0.90)
print('=== NOUVEAU MODÈLE ==='); print(summarize(new))

## 9. Comparer baseline vs nouveau

In [ ]:
def _m(d): b=d['box']; s=d.get('seg',{}); return b['map50'], b['recall'], s.get('map50')
bb=_m(baseline); nn=_m(new)
print(f"{'':12} {'box.mAP50':>10} {'box.recall':>11} {'seg.mAP50':>10}")
print(f"{'baseline':12} {bb[0]!s:>10} {bb[1]!s:>11} {bb[2]!s:>10}")
print(f"{'nouveau':12} {nn[0]!s:>10} {nn[1]!s:>11} {nn[2]!s:>10}")
for name in new.get('per_class', {}):
    m = new['per_class'][name]; print(f"  [{name}] mAP50={m['map50']} recall={m['recall']}")
print('\nRappel : cible recall élevé (ne pas rater de lésion). Seuil recommandé dans le manifeste.')

## 10. Publier l'artefact (si amélioration)
Publie `yolov8_seg.pt` + `manifest.json` sur le repo modèle HF, seulement si le nouveau modèle
améliore le mAP50. La **porte de validation clinique SaMD** reste distincte (`validated` géré à part).

In [ ]:
from mlops.registry import publish_artifact
improved = (new['box']['map50'] or 0) > (baseline['box']['map50'] or 0)
if not improved:
    print(f"⛔ Pas d'amélioration (nouveau {new['box']['map50']} <= baseline {baseline['box']['map50']}) — publication annulée.")
else:
    url = publish_artifact('/content/artifacts/mammo-det', HF_MODEL_REPO, VERSION)
    print('✅ Publié :', url, '(tag', VERSION, ')')

## Suite (hors notebook)
- **Porte de validation clinique** (SaMD) avant `validated: true`.
- **Consommation produit** : déposer `yolov8_seg.pt` sur le Space ml-service (`deploy-ml-space.yml`).
- Régler le **seuil de confiance** (CONFIDENCE_THRESHOLD dans `app/segmentation.py`) sur le
  `recommended_conf_for_recall` du manifeste.
- ⚠️ Vider les sorties du notebook avant tout commit (aucune donnée patient dans git).